# Qwen3-8B MAP Training — Enhanced with Advanced Techniques

This notebook builds upon the QLoRA baseline and adds proven training techniques:

| Technique | Effect | Expected Gain |
|-----------|--------|---------------|
| **AWP** | Adversarial Weight Perturbation → flatter loss surface | +1.0-2.0% |
| **R-Drop** | Consistency regularization via dual forward + KL loss | +0.5-1.5% |
| **EMA** | Exponential Moving Average of weights | +0.3-0.8% |
| **Label Smoothing** | Prevents overconfident predictions | +0.5-1.5% |
| **Layerwise LR Decay** | Lower LR for earlier layers | +0.5-1.0% |
| **Cosine Schedule** | Smooth convergence | +0.3-0.8% |

### References
1. Wu et al. (2020) - AWP: Adversarial Weight Perturbation. NeurIPS 2020
2. Wu et al. (2021) - R-Drop: Regularized Dropout. NeurIPS 2021
3. Polyak & Juditsky (1992) - Exponential Moving Average
4. Szegedy et al. (2016) - Label Smoothing. CVPR 2016

**Output:** LoRA adapter only (not full model) → ~200MB

In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================

!pip install -q -U "transformers>=4.55.0" "accelerate>=0.34.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" datasets scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 42.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# ============================================================
# CELL 2: Imports and Environment Setup
# ============================================================

import gc
import json
import math
import os
import random
import sys
import time as _time
from pathlib import Path
from collections import defaultdict

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import Dataset
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

IN_KAGGLE = os.path.exists("/kaggle/input")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Transformers version:", transformers.__version__)

Torch version: 2.9.0+cu126
CUDA available: True
CUDA device count: 1
GPU: NVIDIA H100 80GB HBM3
GPU Memory: 85.0 GB
Transformers version: 5.3.0


In [3]:
# ============================================================
# CELL 3: Configuration (SAME AS BASELINE)
# ============================================================

BASE_MODEL_NAME = "Qwen/Qwen3-8B"

# Data paths (same as baseline - auto-detection)
TRAIN_CANDIDATES = [
    "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings/train.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings-csv/train.csv",
]
TEST_CANDIDATES = [
    "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings/test.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings-csv/test.csv",
]

def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

TRAIN_CSV = first_existing(TRAIN_CANDIDATES)
TEST_CSV = first_existing(TEST_CANDIDATES)

# Output directory (same structure as baseline)
if IN_KAGGLE:
    SAVE_ROOT = "/kaggle/working/qwen3-8b-qlora"
else:
    SAVE_ROOT = "./qwen3-8b-qlora"

OUTPUT_DIR = Path(SAVE_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# QUICK RUN for testing
# -------------------------
QUICK_RUN = False
QUICK_RUN_TRAIN_SAMPLES = 2048
QUICK_RUN_VAL_SAMPLES = 512

# -------------------------
# Training hyperparameters (SAME AS BASELINE)
# -------------------------
MAX_LENGTH = 256
VAL_SIZE = 0.10
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# 4-bit QLoRA (SAME AS BASELINE)
USE_4BIT = True
USE_DOUBLE_QUANT = True
QUANT_TYPE = "nf4"
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# LoRA config (SAME AS BASELINE)
LORA_R = 64
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# -------------------------
# NEW: Advanced Training Techniques
# -------------------------
# Label Smoothing
LABEL_SMOOTHING = 0.1

# Layerwise Learning Rate Decay
USE_LAYERWISE_LR_DECAY = True
LAYERWISE_LR_DECAY = 0.95  # Each layer gets 0.95x the learning rate of the layer above

# AWP (Adversarial Weight Perturbation)
USE_AWP = True
AWP_START_EPOCH = 1       # Start AWP after first epoch (warmup)
AWP_LR = 0.1              # Perturbation learning rate
AWP_EPS = 1e-4            # Perturbation epsilon

# R-Drop (Consistency Regularization)
USE_RDROP = True
RDROP_ALPHA = 0.5         # Weight for KL divergence loss

# EMA (Exponential Moving Average)
USE_EMA = True
EMA_DECAY = 0.999

# Time management
TIME_LIMIT_HOURS = 11.5   # Leave buffer for Kaggle 12hr limit
PIPELINE_START_TIME = _time.time()

# Apply QUICK_RUN overrides
if QUICK_RUN:
    NUM_EPOCHS = 1
    USE_AWP = False  # Skip AWP in quick run to save time
    OUTPUT_DIR = OUTPUT_DIR / "quick_run"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Mode: {'QUICK RUN' if QUICK_RUN else 'FULL TRAINING'}")
print(f"Model: {BASE_MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS} | LR: {LEARNING_RATE}")
print(f"Batch: {TRAIN_BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"\nAdvanced Techniques:")
print(f"  Label Smoothing: {LABEL_SMOOTHING}")
print(f"  Layerwise LR Decay: {USE_LAYERWISE_LR_DECAY} ({LAYERWISE_LR_DECAY})")
print(f"  AWP: {USE_AWP} (start epoch {AWP_START_EPOCH})")
print(f"  R-Drop: {USE_RDROP} (alpha={RDROP_ALPHA})")
print(f"  EMA: {USE_EMA} (decay={EMA_DECAY})")
print(f"\nTRAIN_CSV: {TRAIN_CSV}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

CONFIGURATION
Mode: FULL TRAINING
Model: Qwen/Qwen3-8B
Epochs: 2 | LR: 0.0002
Batch: 16 x 1 = 16

Advanced Techniques:
  Label Smoothing: 0.1
  Layerwise LR Decay: True (0.95)
  AWP: True (start epoch 1)
  R-Drop: True (alpha=0.5)
  EMA: True (decay=0.999)

TRAIN_CSV: /kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv
OUTPUT_DIR: /kaggle/working/qwen3-8b-qlora


In [4]:
# ============================================================
# CELL 4: Advanced Training Components
# ============================================================

# --- AWP (Adversarial Weight Perturbation) ---
# Reference: Wu et al., NeurIPS 2020, arXiv:2004.05884
class AWP:
    """
    Perturbs model weights in the adversarial direction after the first backward pass.
    The model then learns to be robust to such perturbations → flatter minima.
    """
    def __init__(self, model, adv_lr=0.1, adv_eps=1e-4):
        self.model = model
        self.adv_lr = adv_lr
        self.adv_eps = adv_eps
        self.backup = {}
        self.backup_eps = {}

    def save(self):
        """Save current weights before perturbation."""
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                grad_norm = torch.norm(param.grad)
                if grad_norm != 0:
                    self.backup_eps[name] = self.adv_eps * param.data.abs() / grad_norm

    def attack_step(self):
        """Perturb weights in adversarial direction."""
        for name, param in self.model.named_parameters():
            if name in self.backup and param.grad is not None:
                norm1 = torch.norm(param.grad)
                if norm1 != 0:
                    r_at = self.adv_lr * param.grad / norm1
                    eps = self.backup_eps.get(name, self.adv_eps)
                    param.data.add_(torch.clamp(r_at, -eps, eps))

    def restore(self):
        """Restore original weights after adversarial step."""
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}
        self.backup_eps = {}


# --- EMA (Exponential Moving Average) ---
# Reference: Polyak & Juditsky, SIAM J. Control Optim., 1992
class EMA:
    """
    Maintains a shadow copy of model weights as exponential moving average.
    Use shadow weights at evaluation time for smoother predictions.
    """
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        # Initialize shadow weights
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        """Update shadow weights with EMA."""
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = (
                    self.decay * self.shadow[name] + (1 - self.decay) * param.data
                )

    def apply_shadow(self):
        """Apply shadow weights for evaluation."""
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        """Restore original weights after evaluation."""
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


# --- R-Drop KL Divergence Loss ---
# Reference: Wu et al., NeurIPS 2021, arXiv:2106.14448
def compute_kl_loss(p, q):
    """
    Symmetric KL divergence between two forward passes.
    Forces consistent outputs despite dropout randomness.
    """
    p_loss = F.kl_div(F.log_softmax(p, dim=-1), F.softmax(q, dim=-1), reduction='batchmean')
    q_loss = F.kl_div(F.log_softmax(q, dim=-1), F.softmax(p, dim=-1), reduction='batchmean')
    return (p_loss + q_loss) / 2


# --- Label Smoothing Cross-Entropy ---
# Reference: Szegedy et al., CVPR 2016, arXiv:1512.00567
class LabelSmoothingCE(nn.Module):
    """
    Soft targets instead of hard one-hot: [ε/(K-1), ε/(K-1), 1-ε, ε/(K-1)].
    Prevents overconfident predictions on training data.
    """
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, labels):
        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Hard targets
        nll_loss = F.nll_loss(log_probs, labels, reduction='mean')
        
        # Smooth targets (uniform distribution)
        smooth_loss = -log_probs.mean(dim=-1).mean()
        
        # Combine
        loss = (1 - self.smoothing) * nll_loss + self.smoothing * smooth_loss
        return loss


print("Advanced training components defined:")
print("  - AWP (Adversarial Weight Perturbation)")
print("  - EMA (Exponential Moving Average)")
print("  - R-Drop KL loss")
print("  - Label Smoothing CE")

Advanced training components defined:
  - AWP (Adversarial Weight Perturbation)
  - EMA (Exponential Moving Average)
  - R-Drop KL loss
  - Label Smoothing CE


In [5]:
# ============================================================
# CELL 5: Data Preparation (SAME AS BASELINE)
# ============================================================

train = pd.read_csv(TRAIN_CSV)
print("Raw train shape:", train.shape)

# Fill missing values
train["Misconception"] = train["Misconception"].fillna("NA")
train["StudentExplanation"] = train["StudentExplanation"].fillna("")
train["QuestionText"] = train["QuestionText"].fillna("")
train["MC_Answer"] = train["MC_Answer"].fillna("")

# Create target label
train["target"] = train["Category"].astype(str) + ":" + train["Misconception"].astype(str)

# Infer correct answers from True_* categories
is_true_family = train["Category"].astype(str).str.startswith("True_")
correct = train.loc[is_true_family].copy()
correct["freq"] = correct.groupby(["QuestionId", "MC_Answer"])["MC_Answer"].transform("count")
correct = correct.sort_values(["QuestionId", "freq"], ascending=[True, False])
correct = correct.drop_duplicates(["QuestionId"])[["QuestionId", "MC_Answer"]]
correct["is_correct"] = 1

train = train.merge(correct, on=["QuestionId", "MC_Answer"], how="left")
train["is_correct"] = train["is_correct"].fillna(0).astype(int)

# Format input text
def format_input(row):
    correct_text = "Yes" if int(row["is_correct"]) == 1 else "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Correct? {correct_text}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

train["text"] = train.apply(format_input, axis=1)

# Encode labels
label_encoder = LabelEncoder()
train["label"] = label_encoder.fit_transform(train["target"])
num_labels = len(label_encoder.classes_)

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print(f"Number of labels: {num_labels}")
print(f"Sample counts per category:")
print(train["Category"].value_counts().head(10))

Raw train shape: (36696, 7)
Number of labels: 65
Sample counts per category:
Category
True_Correct           14802
False_Misconception     9457
False_Neither           6542
True_Neither            5265
True_Misconception       403
False_Correct            227
Name: count, dtype: int64


In [6]:
# ============================================================
# CELL 6: Train/Validation Split (SAME AS BASELINE)
# ============================================================

stratify_labels = train["label"] if train["label"].value_counts().min() >= 2 else None

train_df, val_df = train_test_split(
    train[["text", "label", "QuestionId"]].copy(),
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=stratify_labels,
)

# Subsample for QUICK_RUN
if QUICK_RUN:
    train_df = train_df.sample(n=min(QUICK_RUN_TRAIN_SAMPLES, len(train_df)), random_state=SEED)
    val_df = val_df.sample(n=min(QUICK_RUN_VAL_SAMPLES, len(val_df)), random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Train samples: 33026
Validation samples: 3670


In [7]:
# ============================================================
# CELL 7: Tokenizer and Dataset
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Create datasets
train_ds = Dataset.from_pandas(train_df[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)

train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(f"Train dataset: {train_ds}")
print(f"Val dataset: {val_ds}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Map:   0%|          | 0/33026 [00:00<?, ? examples/s]

Map:   0%|          | 0/3670 [00:00<?, ? examples/s]

Train dataset: Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 33026
})
Val dataset: Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 3670
})


In [8]:
# ============================================================
# CELL 8: Load Model with QLoRA (SAME AS BASELINE)
# ============================================================

assert torch.cuda.is_available(), "GPU required!"

# 4-bit quantization config
quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=USE_DOUBLE_QUANT,
        bnb_4bit_quant_type=QUANT_TYPE,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

# Load base model
print("Loading base model...")
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=COMPUTE_DTYPE,
    quantization_config=quantization_config,
    device_map={"":0},
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.problem_type = "single_label_classification"

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

# Find LoRA target modules
def find_target_modules(model):
    candidates = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    found = set()
    for name, _ in model.named_modules():
        last = name.split(".")[-1]
        if last in candidates:
            found.add(last)
    return sorted(found) if found else ["q_proj", "v_proj"]

def find_modules_to_save(model):
    candidates = ["score", "classifier"]
    return [c for c in candidates if any(n.endswith(c) for n, _ in model.named_modules())] or None

target_modules = find_target_modules(model)
modules_to_save = find_modules_to_save(model)

print(f"LoRA target modules: {target_modules}")
print(f"Modules to save: {modules_to_save}")

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    modules_to_save=modules_to_save,
    bias="none",
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

print_trainable_parameters(model)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-8B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Modules to save: ['score']
Trainable: 174,854,144 / 4,270,642,176 (4.09%)


In [9]:
# ============================================================
# CELL 9: Optimizer with Layerwise LR Decay
# ============================================================

def get_layerwise_lr_decay_params(model, base_lr, decay_rate=0.95):
    """
    Apply layerwise learning rate decay: earlier layers get lower LR.
    This helps preserve pretrained knowledge in lower layers.
    """
    if not USE_LAYERWISE_LR_DECAY:
        return [{"params": [p for p in model.parameters() if p.requires_grad], "lr": base_lr}]
    
    # Group parameters by layer depth
    param_groups = []
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    
    # Find max layer number
    layer_nums = []
    for name, _ in model.named_parameters():
        if ".layers." in name:
            try:
                layer_num = int(name.split(".layers.")[1].split(".")[0])
                layer_nums.append(layer_num)
            except:
                pass
    
    max_layer = max(layer_nums) if layer_nums else 0
    
    # Group by layer
    layer_params = defaultdict(list)
    other_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        
        if ".layers." in name:
            try:
                layer_num = int(name.split(".layers.")[1].split(".")[0])
                layer_params[layer_num].append((name, param))
            except:
                other_params.append((name, param))
        else:
            other_params.append((name, param))
    
    # Create param groups with decayed LR
    for layer_num in sorted(layer_params.keys()):
        # Decay: layer 0 gets lowest LR, layer N gets base_lr
        layer_lr = base_lr * (decay_rate ** (max_layer - layer_num))
        
        # Split into decay vs no-decay
        decay_params = [p for n, p in layer_params[layer_num] if not any(nd in n for nd in no_decay)]
        no_decay_params = [p for n, p in layer_params[layer_num] if any(nd in n for nd in no_decay)]
        
        if decay_params:
            param_groups.append({"params": decay_params, "lr": layer_lr, "weight_decay": WEIGHT_DECAY})
        if no_decay_params:
            param_groups.append({"params": no_decay_params, "lr": layer_lr, "weight_decay": 0.0})
    
    # Other params (embeddings, classification head) get base LR
    if other_params:
        decay_params = [p for n, p in other_params if not any(nd in n for nd in no_decay)]
        no_decay_params = [p for n, p in other_params if any(nd in n for nd in no_decay)]
        
        if decay_params:
            param_groups.append({"params": decay_params, "lr": base_lr, "weight_decay": WEIGHT_DECAY})
        if no_decay_params:
            param_groups.append({"params": no_decay_params, "lr": base_lr, "weight_decay": 0.0})
    
    print(f"Created {len(param_groups)} parameter groups with layerwise LR decay")
    print(f"LR range: {base_lr * (decay_rate ** max_layer):.2e} to {base_lr:.2e}")
    
    return param_groups


# Create optimizer
param_groups = get_layerwise_lr_decay_params(model, LEARNING_RATE, LAYERWISE_LR_DECAY)
optimizer = torch.optim.AdamW(param_groups, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Create scheduler
num_training_steps = len(train_ds) // TRAIN_BATCH_SIZE * NUM_EPOCHS // GRAD_ACCUM_STEPS
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

Created 37 parameter groups with layerwise LR decay
LR range: 3.32e-05 to 2.00e-04
Training steps: 4128
Warmup steps: 123


In [10]:
# ============================================================
# CELL 10: MAP@3 Metric
# ============================================================

def compute_map3(logits, labels):
    """
    Compute MAP@3 (Mean Average Precision at 3).
    """
    if isinstance(logits, torch.Tensor):
        logits = logits.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    top3 = np.argsort(-probs, axis=1)[:, :3]
    
    score = 0.0
    for i, y_true in enumerate(labels):
        if top3[i, 0] == y_true:
            score += 1.0
        elif top3[i, 1] == y_true:
            score += 0.5
        elif top3[i, 2] == y_true:
            score += 1.0 / 3.0
    
    return score / len(labels)

print("MAP@3 metric defined.")

MAP@3 metric defined.


In [11]:
# ============================================================
# CELL 11: Enhanced Training Loop
# ============================================================

print("="*70)
print("STARTING ENHANCED TRAINING")
print("="*70)

# Create data loaders
train_loader = DataLoader(
    train_ds, 
    batch_size=TRAIN_BATCH_SIZE, 
    shuffle=True, 
    collate_fn=data_collator,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds, 
    batch_size=EVAL_BATCH_SIZE, 
    shuffle=False, 
    collate_fn=data_collator,
    pin_memory=True,
)

# Loss function
criterion = LabelSmoothingCE(LABEL_SMOOTHING) if LABEL_SMOOTHING > 0 else nn.CrossEntropyLoss()

# Initialize advanced components
awp = AWP(model, AWP_LR, AWP_EPS) if USE_AWP else None
ema = EMA(model, EMA_DECAY) if USE_EMA else None

# Training metrics
best_map3 = 0.0
train_losses = []
val_map3_scores = []

# Training loop
for epoch in range(NUM_EPOCHS):
    epoch_start = _time.time()
    
    # Check time limit
    elapsed_hours = (_time.time() - PIPELINE_START_TIME) / 3600
    if elapsed_hours > TIME_LIMIT_HOURS:
        print(f"\nTime limit reached ({elapsed_hours:.1f}h > {TIME_LIMIT_HOURS}h). Stopping early.")
        break
    
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch + 1}/{NUM_EPOCHS}")
    print(f"{'='*60}")
    
    model.train()
    epoch_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(pbar):
        # Move to device
        batch = {k: v.to(model.device) for k, v in batch.items()}
        labels = batch.pop("labels")
        
        # Forward pass 1
        outputs1 = model(**batch)
        logits1 = outputs1.logits
        loss = criterion(logits1, labels)
        
        # R-Drop: Forward pass 2 with different dropout mask
        if USE_RDROP:
            outputs2 = model(**batch)
            logits2 = outputs2.logits
            loss2 = criterion(logits2, labels)
            
            # KL divergence between two passes
            kl_loss = compute_kl_loss(logits1, logits2)
            
            # Combined loss: average CE + alpha * KL
            loss = (loss + loss2) / 2 + RDROP_ALPHA * kl_loss
        
        # Scale for gradient accumulation
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()
        accum_loss += loss.item()
        
        # Gradient accumulation step
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            
            # AWP: Adversarial step (after warmup epoch)
            if USE_AWP and epoch >= AWP_START_EPOCH:
                awp.save()
                awp.attack_step()
                
                # Forward with perturbed weights
                outputs_adv = model(**batch)
                loss_adv = criterion(outputs_adv.logits, labels) / GRAD_ACCUM_STEPS
                loss_adv.backward()
                
                awp.restore()
            
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # EMA update
            if USE_EMA:
                ema.update()
            
            epoch_loss += accum_loss
            pbar.set_postfix({"loss": f"{accum_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            accum_loss = 0.0
    
    avg_train_loss = epoch_loss / (len(train_loader) // GRAD_ACCUM_STEPS)
    train_losses.append(avg_train_loss)
    
    # ===================== VALIDATION =====================
    print(f"\nValidating...")
    
    # Apply EMA weights for evaluation
    if USE_EMA:
        ema.apply_shadow()
    
    model.eval()
    all_logits = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            labels = batch.pop("labels")
            
            outputs = model(**batch)
            all_logits.append(outputs.logits.cpu())
            all_labels.append(labels.cpu())
    
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    # Compute MAP@3
    val_map3 = compute_map3(all_logits, all_labels)
    val_map3_scores.append(val_map3)
    
    # Restore original weights
    if USE_EMA:
        ema.restore()
    
    epoch_time = (_time.time() - epoch_start) / 60
    
    print(f"\n  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val MAP@3:  {val_map3:.6f}")
    print(f"  Epoch Time: {epoch_time:.1f} min")
    print(f"  LR: {scheduler.get_last_lr()[0]:.2e}")
    
    # Save best model
    if val_map3 > best_map3:
        best_map3 = val_map3
        print(f"  *** New best MAP@3: {best_map3:.6f} ***")
        
        # Apply EMA for saving
        if USE_EMA:
            ema.apply_shadow()
        
        # Save adapter
        model.save_pretrained(str(OUTPUT_DIR / "adapter"))
        tokenizer.save_pretrained(str(OUTPUT_DIR / "adapter"))
        
        if USE_EMA:
            ema.restore()

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best MAP@3: {best_map3:.6f}")
print(f"Total time: {(_time.time() - PIPELINE_START_TIME) / 60:.1f} min")

STARTING ENHANCED TRAINING

EPOCH 1/2


Epoch 1:   0%|          | 0/2065 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 1: 100%|██████████| 2065/2065 [35:54<00:00,  1.04s/it, loss=0.7650, lr=1.74e-05]



Validating...


Validation: 100%|██████████| 115/115 [00:31<00:00,  3.70it/s]



  Train Loss: 1.2373
  Val MAP@3:  0.938283
  Epoch Time: 36.4 min
  LR: 1.74e-05
  *** New best MAP@3: 0.938283 ***

EPOCH 2/2


Epoch 2: 100%|██████████| 2065/2065 [55:21<00:00,  1.61s/it, loss=0.7975, lr=2.04e-11]



Validating...


Validation: 100%|██████████| 115/115 [00:31<00:00,  3.70it/s]



  Train Loss: 0.9351
  Val MAP@3:  0.946912
  Epoch Time: 55.9 min
  LR: 2.04e-11
  *** New best MAP@3: 0.946912 ***

TRAINING COMPLETE!
Best MAP@3: 0.946912
Total time: 92.8 min


In [12]:
# ============================================================
# CELL 12: Save Final Artifacts
# ============================================================

# Apply EMA for final save
if USE_EMA:
    ema.apply_shadow()

# Save adapter (may already be saved as best, but ensure final state)
final_adapter_path = OUTPUT_DIR / "adapter"
model.save_pretrained(str(final_adapter_path))
tokenizer.save_pretrained(str(final_adapter_path))

# Save label encoder
joblib.dump(label_encoder, str(OUTPUT_DIR / "label_encoder.joblib"))

# Save training config
config_to_save = {
    "base_model_name": BASE_MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_labels": num_labels,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "use_4bit": USE_4BIT,
    "quant_type": QUANT_TYPE,
    "compute_dtype": str(COMPUTE_DTYPE),
    "val_size": VAL_SIZE,
    "seed": SEED,
    # Advanced techniques
    "label_smoothing": LABEL_SMOOTHING,
    "use_layerwise_lr_decay": USE_LAYERWISE_LR_DECAY,
    "layerwise_lr_decay": LAYERWISE_LR_DECAY,
    "use_awp": USE_AWP,
    "awp_start_epoch": AWP_START_EPOCH,
    "awp_lr": AWP_LR,
    "awp_eps": AWP_EPS,
    "use_rdrop": USE_RDROP,
    "rdrop_alpha": RDROP_ALPHA,
    "use_ema": USE_EMA,
    "ema_decay": EMA_DECAY,
    # Results
    "best_map3": best_map3,
    "train_losses": train_losses,
    "val_map3_scores": val_map3_scores,
}

with open(str(OUTPUT_DIR / "run_config.json"), "w") as f:
    json.dump(config_to_save, f, indent=2)

print("\nSaved artifacts:")
print(f"  Adapter: {final_adapter_path}")
print(f"  Label encoder: {OUTPUT_DIR / 'label_encoder.joblib'}")
print(f"  Config: {OUTPUT_DIR / 'run_config.json'}")

# List output files
print(f"\nOutput directory contents:")
for f in OUTPUT_DIR.rglob("*"):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(OUTPUT_DIR)}: {size_mb:.1f} MB")


Saved artifacts:
  Adapter: /kaggle/working/qwen3-8b-qlora/adapter
  Label encoder: /kaggle/working/qwen3-8b-qlora/label_encoder.joblib
  Config: /kaggle/working/qwen3-8b-qlora/run_config.json

Output directory contents:
  run_config.json: 0.0 MB
  label_encoder.joblib: 0.0 MB
  adapter/chat_template.jinja: 0.0 MB
  adapter/adapter_model.safetensors: 699.5 MB
  adapter/tokenizer_config.json: 0.0 MB
  adapter/tokenizer.json: 11.4 MB
  adapter/adapter_config.json: 0.0 MB
  adapter/README.md: 0.0 MB


In [13]:
# ============================================================
# CELL 13: Sanity Check Inference
# ============================================================

print("\nSanity check inference...")

model.eval()

# Get a sample
sample_text = val_df.loc[0, "text"]
sample_label = val_df.loc[0, "label"]

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

top3_idx = torch.topk(logits, k=3, dim=-1).indices[0].cpu().numpy()
top3_labels = label_encoder.inverse_transform(top3_idx)
true_label = label_encoder.inverse_transform([sample_label])[0]

print(f"\nSample text (truncated):\n{sample_text[:200]}...")
print(f"\nTrue label: {true_label}")
print(f"\nTop-3 predictions:")
for rank, label in enumerate(top3_labels, 1):
    marker = "✓" if label == true_label else " "
    print(f"  {rank}. {label} {marker}")


Sanity check inference...

Sample text (truncated):
Question: The probability of an event occurring is \( 0.9 \).

Which of the following most accurately describes the likelihood of the event occurring?
Answer: Likely
Correct? Yes
Student Explanation...

True label: True_Correct:NA

Top-3 predictions:
  1. True_Correct:NA ✓
  2. True_Neither:NA  
  3. False_Misconception:Scale  


In [14]:
# ============================================================
# CELL 14: Training Summary
# ============================================================

print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)

total_time_min = (_time.time() - PIPELINE_START_TIME) / 60
total_time_hrs = total_time_min / 60

summary = f"""
Model: {BASE_MODEL_NAME}
QLoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}

Training:
  - Epochs: {NUM_EPOCHS}
  - Batch: {TRAIN_BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS}
  - Learning Rate: {LEARNING_RATE}
  - Total Time: {total_time_min:.1f} min ({total_time_hrs:.2f} hrs)

Advanced Techniques:
  - Label Smoothing: {LABEL_SMOOTHING}
  - Layerwise LR Decay: {USE_LAYERWISE_LR_DECAY} ({LAYERWISE_LR_DECAY})
  - AWP: {USE_AWP} (start epoch {AWP_START_EPOCH})
  - R-Drop: {USE_RDROP} (alpha={RDROP_ALPHA})
  - EMA: {USE_EMA} (decay={EMA_DECAY})

Results:
  - Best MAP@3: {best_map3:.6f}
  - Train losses: {[f'{l:.4f}' for l in train_losses]}
  - Val MAP@3: {[f'{s:.6f}' for s in val_map3_scores]}

Output:
  - Adapter: {OUTPUT_DIR / 'adapter'}
  - Label encoder: {OUTPUT_DIR / 'label_encoder.joblib'}
  - Config: {OUTPUT_DIR / 'run_config.json'}
"""

print(summary)

# Save summary
with open(str(OUTPUT_DIR / "training_summary.txt"), "w") as f:
    f.write(summary)

print("\n" + "="*70)
print("NOTEBOOK COMPLETE!")
print("="*70)


TRAINING SUMMARY

Model: Qwen/Qwen3-8B
QLoRA: r=64, alpha=32, dropout=0.05

Training:
  - Epochs: 2
  - Batch: 16 x 1 = 16
  - Learning Rate: 0.0002
  - Total Time: 92.8 min (1.55 hrs)

Advanced Techniques:
  - Label Smoothing: 0.1
  - Layerwise LR Decay: True (0.95)
  - AWP: True (start epoch 1)
  - R-Drop: True (alpha=0.5)
  - EMA: True (decay=0.999)

Results:
  - Best MAP@3: 0.946912
  - Train losses: ['1.2373', '0.9351']
  - Val MAP@3: ['0.938283', '0.946912']

Output:
  - Adapter: /kaggle/working/qwen3-8b-qlora/adapter
  - Label encoder: /kaggle/working/qwen3-8b-qlora/label_encoder.joblib
  - Config: /kaggle/working/qwen3-8b-qlora/run_config.json


NOTEBOOK COMPLETE!
